# PyTorch MLP on EEG bandpower features

**What this notebook is.** A demo training a small set of EEG data. The pipeline attempts to mirror the classifiers in `src/analysis/`, but using PyTorch for Neuronal Network classification.

**Disclaimer:** The original sklearn classifiers were trained on lab-internal BioMag data (results reported in a separate thesis). That data isn't available due to data privacy constraints, so this notebook uses MNE's built-in `eegbci` motor-imagery dataset.


**To run this notebook.** Activate the conda env (`conda activate mtbi_meeg_conda`), pick that kernel in VS Code or `jupyter`, and Run All. First execution might be slow since it downloads ~5 MB of data into `~/mne_data/`.

Resources:
https://github.com/rasbt/machine-learning-book?
https://docs.pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html
Check https://www.youtube.com/watch?v=JHWqWIoac2I

## 1. Environment setup

import libraries and define seed and other things

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Load data

We use 20 subjects, 2 runs each from the EEG Motor Movement/Imagery Dataset, sampled at 160Hz and epochs of 1min
- Run 1: baseline eyes-open (label 0)
- Run 2: baseline eyes-closed (label 1)

NOTE: loading the data might take up to 10min

In [4]:
N_SUBJECTS = 20
RUNS = [1, 2]  # 1 = eyes open, 2 = eyes closed

# Load data function as 3-uples [data, eyes, sampling_freq]
def load_subject(subject_id: int) -> list[tuple[np.ndarray, int, float]]:
    """Load both runs for one subject. Returns list of (raw_data, label, sfreq)."""
    paths = mne.datasets.eegbci.load_data(subject_id, RUNS, update_path=True)
    out = []
    for path, label in zip(paths, [0, 1]):
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        out.append((raw.get_data(), label, raw.info['sfreq']))
    return out

# Iterate over subjects and extend into a 1D array of 3-uples
samples = []
for subj in range(1, N_SUBJECTS + 1):
    samples.extend(load_subject(subj))

print(f'Loaded {len(RUNS)} runs for each of the {N_SUBJECTS} subjects')
print(f'Shape of first sample: {samples[0][0].shape}  (channels x timepoints)')

Loaded 2 runs for each of the 20 subjects
Shape of first sample: (64, 9760)  (channels x timepoints)


## 3. Feature extraction — bandpower

Following the same approach as `src/processing/04_bandpower.py`: for each run, compute the average power in the EEG frequency bands per channel. This yields a `(n_channels × n_bands)` feature matrix per run, which we flatten to a 1D vector.

Bands used here: delta (1-4 Hz), theta (4-8), alpha (8-13), beta (13-30), gamma (30-45). Standard clinical EEG bands.

In [ ]:
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

# Naming convention reminder
# - UPPER_CASE: constants, don't change after definition.
# e.g. SEED, N_SUBJECTS, RUNS,  BANDS
# - snake_case: regular variables and functions
# e.g. samples, psd, freqs, n_channels, features, bandpower_features

def bandpower_features()
    """Compute bandpower per channel per band, return flattened (n_channels * n_bands,) vector."""
    return features
                                     



## 4. Train / test split

20 subjects and a 80/20 split? 85/15?

In [ ]:
# Remember that same subject must have both runs in the same group

## 5. Model

A 2 layer MLP? Input would be the bandpower feature vector (n_channels × n_bands), output is a binary classification.




In [ ]:
class MLP(nn.Module):
    # Define model architecture
    def __init__(self, input_dim=320, hidden_dim=64, out_features=1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_dim, out_features)
    # Forward method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP(input_dim=X_train.shape[1]).to(DEVICE)
print(model)

## 6. Training loop

In [ ]:
#### WHAT ARE THESE?
N_EPOCHS = 100
LR = 1e-3



## 7. Evaluation or output?

Final test accuracy and ROC AUC

In [ ]:
# model.eval()
# smth

## 9. Next steps

What would come next if this had more time:
- Hopefully this is more or less successful 
- Better understanding on some of the choices (n_epochs, data structure, etc)
- Implement k-fold CV
- Run on real mTBI data: e.g., the one from BioMag used for the sklearn classifiers or an open source one like OpenNeuro `ds005114`
- Compare performance against the sklearn classifiers on the same dataset used for them in the thesis.